In [1]:
import asyncio
from collections import defaultdict
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


In [2]:
import sys
from pathlib import Path

# Asumiendo que notebooks/ está dentro de la raíz del proyecto
root_path = Path().resolve().parent  # sube un nivel
sys.path.append(str(root_path))

In [3]:
from agents.classifier_agent import ClassifierAgent
from agents.aggregator_agent import AggregatorAgent
from data.dataset_registry import DatasetRegistry
from data.loaders.sklearn_loader import SklearnLoader

In [5]:
import asyncio
import numpy as np
import torch

from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier

from data.dataset_registry import DatasetRegistry
from data.loaders.sklearn_loader import SklearnLoader

from agents.classifier_agent import ClassifierAgent
from agents.aggregator_agent import AggregatorAgent

from explainers.shap_explainer import ShapExplainer

from models.sklearn_model import SklearnModel
from models.torch_model import TorchModel
from models.random_forest import RandomForest as AdvancedRF

# Visualización (todo en un solo módulo como acordamos)
from visualization.visualization import (
    plot_metrics_over_time,
    plot_explanation_similarity,
    plot_explanation_divergence,
    plot_agents_dashboard
)

import torch.nn as nn
from models.torch_model import TorchModel

class IrisMLP(nn.Module):
    def __init__(self, input_dim=4, hidden_dim=16, num_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        return self.net(x)



async def run_test():
    print("\n==================== TEST MULTI-AGENT ====================")

    # ------------------- Dataset -------------------
    registry = DatasetRegistry()
    dataset_id = "iris"
    registry.register(dataset_id, SklearnLoader(load_iris))

    X, y, meta = registry.load(dataset_id)
    print(f"[Dataset] {dataset_id} cargado → X={X.shape}, y={y.shape}")

    instance = X[:1]

    # ------------------- Modelos -------------------
    rf_base = SklearnModel(
        RandomForestClassifier(n_estimators=20, random_state=0)
    )

    rf_adv = AdvancedRF(
        n_estimators=30,
        max_depth=5,
        random_state=1
    )

    torch_model = TorchModel(
        nn_model=IrisMLP(input_dim=X.shape[1]),
        lr=1e-3,
        epochs=20,
        batch_size=16
    )

    # ------------------- Clasificadores -------------------
    classifiers = {
        "rf_base": ClassifierAgent(
            agent_id="rf_base",
            model=rf_base,
            explainers=[ShapExplainer()],
            dataset_id=dataset_id,
            registry=registry
        ),
        "rf_adv": ClassifierAgent(
            agent_id="rf_adv",
            model=rf_adv,
            explainers=[ShapExplainer()],
            dataset_id=dataset_id,
            registry=registry
        ),
        "nn_torch": ClassifierAgent(
            agent_id="nn_torch",
            model=torch_model,
            explainers=[ShapExplainer()],  # luego puedes cambiar por DeepSHAP
            dataset_id=dataset_id,
            registry=registry
        )
    }

    classifier_ids = list(classifiers.keys())

    # ------------------- Aggregator -------------------
    aggregator = AggregatorAgent(
        classifier_ids=classifier_ids,
        max_iterations=2,   # 🔴 solo 2 iteraciones para depuración
        alpha=0.4,
        beta=0.15,
        gamma=0.25
    )

    # ------------------- Queues -------------------
    queues = {cid: agent.inbox for cid, agent in classifiers.items()}
    queues["aggregator"] = aggregator.inbox

    # ------------------- Setup -------------------
    print("\n[Setup] Inicializando agentes...\n")
    await asyncio.gather(*(agent.setup() for agent in classifiers.values()))

    # ------------------- Tasks -------------------
    tasks = [
        asyncio.create_task(agent.run(queues))
        for agent in classifiers.values()
    ]
    tasks.append(
        asyncio.create_task(
            aggregator.run(queues, instance)
        )
    )

    await asyncio.gather(*tasks)

    print("\n==================== TEST FINALIZADO ✅ ====================\n")

    # =========================================================
    # VISUALIZACIÓN
    # =========================================================

    print("\n[Visualización] Métricas por agente\n")

    for agent in classifiers.values():
        plot_metrics_over_time(agent.metrics_history, metric_name="accuracy")
        plot_metrics_over_time(agent.metrics_history, metric_name="f1")

        plot_explanation_similarity(agent)

    print("\n[Visualización] Divergencia global de explicaciones\n")
    plot_explanation_divergence(classifiers)

    print("\n[Visualización] Dashboard global\n")
    plot_agents_dashboard(classifiers)


In [6]:
await run_test()



==================== TEST MULTI-AGENT ====================
[Dataset] iris cargado → X=(150, 4), y=(150,)

[Setup] Inicializando agentes...

[rf_base] Setup iniciado
[rf_adv] Setup iniciado
[nn_torch] Setup iniciado


RuntimeError: El modelo no ha sido entrenado